# SVM Classification with PCA

Dataset: Wisconsin Diagnostic Breast Cancer (UCI)

Source: https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/wdbc.data

In [ ]:
%pip -q install pandas numpy matplotlib seaborn scikit-learn

# 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

RANDOM_STATE = 42
pd.set_option("display.max_columns", 40)
sns.set_theme(style="whitegrid")

# 2. Load Dataset

In [ ]:
data_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/wdbc.data"

feature_names = [
    "radius_mean", "texture_mean", "perimeter_mean", "area_mean", "smoothness_mean",
    "compactness_mean", "concavity_mean", "concave_points_mean", "symmetry_mean",
    "fractal_dimension_mean", "radius_se", "texture_se", "perimeter_se", "area_se",
    "smoothness_se", "compactness_se", "concavity_se", "concave_points_se",
    "symmetry_se", "fractal_dimension_se", "radius_worst", "texture_worst",
    "perimeter_worst", "area_worst", "smoothness_worst", "compactness_worst",
    "concavity_worst", "concave_points_worst", "symmetry_worst", "fractal_dimension_worst"
]

columns = ["id", "diagnosis"] + feature_names
df = pd.read_csv(data_url, header=None, names=columns)
df["diagnosis"] = df["diagnosis"].map({"B": 0, "M": 1})

print(f"Dataset shape: {df.shape}")
display(df.head())

# 3. Exploratory Data Analysis (EDA)

In [ ]:
print("Feature names:")
print(feature_names)

print("\nData types:")
display(df.dtypes.rename("dtype").to_frame())

print("\nDescriptive statistics:")
display(df[feature_names].describe().T)

print("\nClass distribution:")
class_counts = df["diagnosis"].map({0: "Benign (B)", 1: "Malignant (M)"}).value_counts()
display(class_counts.rename("count").to_frame())

ax = sns.countplot(data=df, x="diagnosis", hue="diagnosis", legend=False)
ax.set_xticks([0, 1], ["Benign (0)", "Malignant (1)"])
ax.set_title("Class distribution")
ax.set_xlabel("Diagnosis")
ax.set_ylabel("Number of samples")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
plot_features = ["radius_mean", "texture_mean", "perimeter_mean", "area_mean", "smoothness_mean", "concavity_mean"]

for ax, feature in zip(axes.flat, plot_features):
    sns.histplot(data=df, x=feature, hue="diagnosis", bins=25, element="step", stat="density", common_norm=False, ax=ax)
    ax.set_title(feature)

plt.tight_layout()
plt.show()

# 4. Data Quality Checks

In [ ]:
print("Missing values by column:")
missing = df.isna().sum()
display(missing[missing > 0].rename("missing_count").to_frame() if missing.any() else pd.DataFrame({"status": ["No missing values found"]}))

print(f"Exact duplicate rows: {df.duplicated().sum()}")

X_all = df[feature_names].copy()
y_all = df["diagnosis"].copy()

constant_features = [col for col in X_all.columns if X_all[col].nunique(dropna=False) <= 1]
print("Constant features:", constant_features if constant_features else "None")

duplicate_feature_pairs = []
for i, col_a in enumerate(X_all.columns):
    for col_b in X_all.columns[i + 1:]:
        if X_all[col_a].equals(X_all[col_b]):
            duplicate_feature_pairs.append((col_a, col_b))
print("Exact duplicate feature pairs:", duplicate_feature_pairs if duplicate_feature_pairs else "None")

feature_matrix_rank = np.linalg.matrix_rank(X_all.to_numpy())
print(f"Numerical matrix rank: {feature_matrix_rank} of {X_all.shape[1]} features")
if feature_matrix_rank < X_all.shape[1]:
    print("The feature matrix has at least one exact linear dependency at numerical rank tolerance.")
else:
    print("No exact linear dependency detected at numerical rank tolerance.")

correlation = X_all.corr()
upper = correlation.where(np.triu(np.ones(correlation.shape), k=1).astype(bool))
high_correlation_pairs = [
    (row, col, upper.loc[row, col])
    for col in upper.columns
    for row in upper.index
    if pd.notna(upper.loc[row, col]) and abs(upper.loc[row, col]) >= 0.999
]
print("Feature pairs with absolute correlation >= 0.999:")
display(pd.DataFrame(high_correlation_pairs, columns=["feature_1", "feature_2", "correlation"]) if high_correlation_pairs else pd.DataFrame({"status": ["None found at this threshold"]}))

plt.figure(figsize=(16, 12))
sns.heatmap(correlation, cmap="coolwarm", center=0, xticklabels=True, yticklabels=True)
plt.title("Correlation heatmap of the 30 numerical features")
plt.tight_layout()
plt.show()

# 5. Split Dataset

In [ ]:
X = df[feature_names].copy()
y = df["diagnosis"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)
print("\nTraining class proportions:")
display(y_train.value_counts(normalize=True).sort_index().rename(index={0: "Benign", 1: "Malignant"}).to_frame("proportion"))
print("\nTesting class proportions:")
display(y_test.value_counts(normalize=True).sort_index().rename(index={0: "Benign", 1: "Malignant"}).to_frame("proportion"))

# 6. SVM Without PCA

In [ ]:
original_model = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf", class_weight="balanced", random_state=RANDOM_STATE))
])

original_model.fit(X_train, y_train)
original_predictions = original_model.predict(X_test)

print(classification_report(y_test, original_predictions, target_names=["Benign", "Malignant"], digits=4))

# 7. PCA and SVM With Reduced Features

In [ ]:
pca_model = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=0.95, svd_solver="full")),
    ("svm", SVC(kernel="rbf", class_weight="balanced", random_state=RANDOM_STATE))
])

pca_model.fit(X_train, y_train)
pca_predictions = pca_model.predict(X_test)

fitted_pca = pca_model.named_steps["pca"]
print(f"Original number of features: {X_train.shape[1]}")
print(f"Number of PCA components retained: {fitted_pca.n_components_}")
print(f"Explained variance retained: {fitted_pca.explained_variance_ratio_.sum():.4%}")
print(f"Variance not retained: {1 - fitted_pca.explained_variance_ratio_.sum():.4%}")

print(classification_report(y_test, pca_predictions, target_names=["Benign", "Malignant"], digits=4))

In [ ]:
pca_for_plot = PCA().fit(StandardScaler().fit_transform(X_train))
cumulative_variance = np.cumsum(pca_for_plot.explained_variance_ratio_)

plt.figure(figsize=(9, 5))
plt.plot(np.arange(1, len(cumulative_variance) + 1), cumulative_variance, marker="o")
plt.axhline(0.95, linestyle="--", label="95% variance")
plt.xlabel("Number of principal components")
plt.ylabel("Cumulative explained variance")
plt.title("PCA explained variance on the training set")
plt.xticks(np.arange(1, len(cumulative_variance) + 1, 2))
plt.legend()
plt.tight_layout()
plt.show()

# 8. Model Evaluation

In [ ]:
def evaluate_predictions(y_true, y_pred, model_name):
    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision (malignant)": precision_score(y_true, y_pred, zero_division=0),
        "Recall (malignant)": recall_score(y_true, y_pred, zero_division=0),
        "F1-score (malignant)": f1_score(y_true, y_pred, zero_division=0)
    }

test_results = pd.DataFrame([
    evaluate_predictions(y_test, original_predictions, "SVM with original features"),
    evaluate_predictions(y_test, pca_predictions, "SVM with PCA (95% variance)")
])

display(test_results.set_index("Model").style.format("{:.4f}"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, original_predictions, display_labels=["Benign", "Malignant"],
    cmap="Blues", values_format="d", ax=axes[0]
)
axes[0].set_title("Original features")
ConfusionMatrixDisplay.from_predictions(
    y_test, pca_predictions, display_labels=["Benign", "Malignant"],
    cmap="Blues", values_format="d", ax=axes[1]
)
axes[1].set_title("PCA features")
plt.tight_layout()
plt.show()

# 9. Cross-Validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {"accuracy": "accuracy", "precision": "precision", "recall": "recall", "f1": "f1"}

original_cv = cross_validate(original_model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
pca_cv = cross_validate(pca_model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)

def summarize_cv(cv_results, model_name):
    return {
        "Model": model_name,
        "CV accuracy (mean)": cv_results["test_accuracy"].mean(),
        "CV accuracy (std)": cv_results["test_accuracy"].std(),
        "CV precision (mean)": cv_results["test_precision"].mean(),
        "CV recall (mean)": cv_results["test_recall"].mean(),
        "CV F1 (mean)": cv_results["test_f1"].mean()
    }

cv_results_table = pd.DataFrame([
    summarize_cv(original_cv, "SVM with original features"),
    summarize_cv(pca_cv, "SVM with PCA (95% variance)")
])

display(cv_results_table.set_index("Model").style.format("{:.4f}"))

# 10. Final Results Comparison

In [ ]:
final_results = test_results.merge(cv_results_table, on="Model")
display(final_results.set_index("Model").style.format({
    "Accuracy": "{:.4f}",
    "Precision (malignant)": "{:.4f}",
    "Recall (malignant)": "{:.4f}",
    "F1-score (malignant)": "{:.4f}",
    "CV accuracy (mean)": "{:.4f}",
    "CV accuracy (std)": "{:.4f}",
    "CV precision (mean)": "{:.4f}",
    "CV recall (mean)": "{:.4f}",
    "CV F1 (mean)": "{:.4f}"
}))

# 11. Conclusion